# Micro-Hay conservative loss from scratch 04

Trains GRU, Branch ELM, causal ConvGRU and causal ConvLSTM from random initialization. Every model first learns the normal 61-state MSE for ten epochs; the new conservative spike terms are then introduced over five epochs. The failed legacy event/BCE objective is not used.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path('/kaggle/working/LearningSingleCompartiment')
if not (ROOT / 'pyproject.toml').exists():
    if ROOT.exists() and any(ROOT.iterdir()): raise RuntimeError(f'{ROOT} exists but is not the project')
    subprocess.check_call(['git', 'clone', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', 'main'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)

## Reuse the saved HDF5
This notebook never intentionally creates a replacement dataset. Mount the previously saved notebook output or Kaggle Dataset through **Add Input**. The cell searches both `/kaggle/working` and `/kaggle/input`; if the HDF5 is absent it stops before training.

In [ ]:
working_dataset = Path('/kaggle/working/hay_micro_4c_event_enriched_v2.h5')
candidates = [working_dataset] if working_dataset.exists() else []
if Path('/kaggle/input').exists():
    candidates += list(Path('/kaggle/input').rglob('hay_micro_4c_event_enriched_v2.h5'))
if not candidates:
    raise FileNotFoundError('Monta con Add Input il dataset HDF5 già generato; questo notebook non lo rigenera.')
DATASET = candidates[0]
print('Dataset riusato:', DATASET, f'({DATASET.stat().st_size/2**30:.2f} GiB)')

In [ ]:
os.environ['HAY_EVENT_DATASET'] = str(DATASET)
os.environ['HAY_EVENT_OUTPUT'] = '/kaggle/working/hay_micro_conservative_scratch_04'
os.environ['HAY_EVENT_MODELS'] = 'gru_conservative,branch_elm,conv_gru,conv_lstm'
os.environ['HAY_EVENT_LOSS'] = 'conservative'
os.environ['HAY_EVENT_MSE_WARMUP_EPOCHS'] = '10'
os.environ['HAY_EVENT_CURRICULUM_EPOCHS'] = '5'
os.environ['HAY_EVENT_EPOCHS'] = '30'
os.environ['HAY_EVENT_REPLAYS'] = '0'
os.environ['HAY_EVENT_WINDOWS_PER_EPOCH'] = '48'
os.environ['HAY_EVENT_REUSE_MODELS'] = '1'
%run /kaggle/working/LearningSingleCompartiment/notebooks/micro_event_aware_training_02.py

The engine writes a best checkpoint and a resumable `*_last.pt` after every completed epoch. Re-running the cell resumes the interrupted architecture and skips completed ones.

In [ ]:
from IPython.display import display
display(pd.read_csv('/kaggle/working/hay_micro_conservative_scratch_04/comparison.csv').sort_values('best_validation_selection_loss'))

In [ ]:
from shutil import make_archive
from IPython.display import FileLink, display
source = Path('/kaggle/working/hay_micro_conservative_scratch_04')
zip_path = Path(make_archive('/kaggle/working/hay_micro_conservative_scratch_04_complete', 'zip', root_dir=source.parent, base_dir=source.name))
print('Archive:', zip_path, f'({zip_path.stat().st_size/2**20:.1f} MiB)')
display(FileLink(str(zip_path)))